# Lesson 07 - नियोजन डिझाइन पॅटर्न

हा नोटबुक माइक्रोसॉफ्ट एजंट फ्रेमवर्क वापरून AI एजंट्ससाठी **नियोजन डिझाइन पॅटर्न** कसे कार्य करते हे दर्शवितो.
आपण जटिल प्रवास मागण्या संरचित उपकार्यांमध्ये कशा विभागायच्या, त्या तज्ञ एजंट्सना कशा नियुक्त करायच्या,
आणि परिणामी नियोजन कसे कार्यान्वित करायचे हे शिकाल — हे सर्व Pydantic मॉडेल्सद्वारे चालवलेल्या संरचित आउटपुटचा वापर करून.


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## कार्य विभाजन

कार्य विभाजन हे नियोजन डिझाइन पॅटर्नचे मूळ आहे. एका एजंटवर संपूर्ण गुंतागुंतीचा विनंती एकट्या पद्धतीने हाताळण्याऐवजी, आपण समस्येला लहान, चांगल्या प्रकारे व्याख्यायित **उपकार्यांमध्ये** विभागतो. प्रत्येक उपकार्य एका तज्ञ एजंटला (उदा., फ्लाइट्स, हॉटेल्स, क्रियाकलाप) स्पष्ट प्राथमिकता आणि अवलंबित्व क्रमाने दिले जाते.

हा दृष्टिकोन अनेक फायदे देतो:
- **स्पष्टता**: प्रत्येक उपकार्याला एकच जबाबदारी असते.
- **समांतरता**: स्वतंत्र उपकार्ये एकाच वेळी चालू शकतात.
- **विश्वसनीयता**: अपयश केवळ वेगळ्या उपकार्यांपुरते मर्यादित असते.
- **बजेट ट्रॅकिंग**: खर्च प्रत्येक उपकार्यासाठी अंदाजित केले जातात आणि एकत्र केले जातात.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## संरचित आउटपुटसह नियोजन एजंट तयार करणे

नियोजन एजंट **फ्रंट डेस्क समन्वयक** म्हणून काम करतो. उच्च-स्तरीय प्रवास विनंती दिल्यास तो एक संरचित `TravelPlan` तयार करतो — विनंतीला उपकार्यांमध्ये विभागणे, प्राधान्य निश्चित करणे, आणि अवलंबित्वे ओळखणे जेणेकरून कन्सियर्ज किंवा अंमलबजावणी स्तर काम करू शकेल.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## तज्ञ साधनांसह योजना अंमलबजावणी

एकदा फ्रंट डेस्क एजंटने रचलेली योजना तयार केली की, **कॉनसिआर्ज एजंट** ती अंमलात आणतो.
प्रत्येक तज्ञ साधन एका उपकार्याच्या श्रेणीस हाताळते (फ्लाइट, हॉटेल, क्रियाकलाप). कॉन्सिआर्ज
योजनेतील उपकार्यांना अवलंबित्वाच्या क्रमाने फिरवतो आणि प्रत्येक उपकार्य योग्य साधनाला पाठवतो.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## सारांश

या धड्यात तुम्ही AI एजंट्ससाठी **योजना डिझाइन पॅटर्न** शिकलात:

1. **कार्य विभाजन** — एक फ्रंट डेस्क योजना एजंट एका गुंतागुंतीच्या प्रवास विनंतीला Pydantic मॉडेल्सचा वापर करून संरचित उपकार्यांमध्ये विभाजित करतो, प्रत्येकाला प्राधान्यक्रम आणि अवलंबित्वांसह तज्ञ एजंटला नेमतो.
2. **संरचित आउटपुट** — `response_format` पास केल्यावर एजंट मुक्त स्वरूपातील मजकूराच्या ऐवजी एक प्रमाणीकरण केलेले `TravelPlan` ऑब्जेक्ट परत करतो, ज्यामुळे पुढील प्रक्रिया विश्वसनीय होते.
3. **योजना अंमलबजावणी** — एक कन्सिआर्ज एजंट उपकार्यांमधून तज्ञ साधने (`book_flight`, `reserve_hotel`, `book_activity`) वापरून योजना अंमलात आणतो आणि निकालांची नोंद करतो.

हा पॅटर्न *काय करायचे आहे* (योजना करणे) आणि *कसे करायचे आहे* (अंमलबजावणी) यांना वेगळे करतो, ज्यामुळे एजंट अधिक मॉड्युलर, तपासण्यास सोपे आणि वाढवण्यास सुलभ होतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI अनुवाद सेवेसह [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्नशील असलो तरी, कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये त्रुटी किंवा अचूकतेच्या दोष असू शकतात. मूळ दस्तऐवज त्याच्या स्थानिक भाषेत अधिकृत स्त्रोत मानला पाहिजे. महत्त्वपूर्ण माहिती साठी व्यावसायिक मानव अनुवादाची शिफारस केली जाते. या अनुवादाच्या वापरामुळे उद्भवलेल्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थ लावण्याबाबत आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
